Question 1: Cubic Histogram Classification Rule
Dataset: banknote_authentication.csv
Description: banknote_authentication.names
1. Select two variance features (Variance, Skewness)
2. Implement and visualize cubic histogram classification with:
a. Equal-width binning (3 bins/axis → 9 cells),
b. Max-count class assignment per cell.
3. Test accuracy with different bin configurations:
a. 2 × 2 bins → 4 cells
b. 4 × 4 bins → 16 cells
4. Analyze:
a. how bin granularity affects overfitting?
b. why this method is impractical for high-dimensional data?

In [23]:
import pandas as pd
import numpy as np

from glob import glob

import IPython.display as ipd
from itertools import cycle

import os
import io
from pydantic import BaseModel, computed_field, Field
from typing import Any, cast
from tqdm import tqdm
import plotly.express as px
import plotly.graph_objects as go
import plotly.colors as pc
from plotly.subplots import make_subplots
from collections import Counter


color_palette = pc.qualitative.Plotly  # or try: D3, G10, T10, Pastel, Bold
color_cycle = cycle(color_palette)

# data foler from current file location
DATA_DIR = os.path.join(os.getcwd(), "../data")
RAW_DIR = os.path.join(DATA_DIR, "raw")
DATASETT_NAME = "banknote_authentication.csv"
DATASETT_PATH = os.path.join(RAW_DIR, DATASETT_NAME)
DATASETT_PATH

'/Users/valiantlynx/projects/pattern-recognition/notebooks/../data/raw/banknote_authentication.csv'

In [24]:
class BankNote(BaseModel):
    variance_of_wavelet_transformed_image_continuous: float = Field(alias='variance')
    skeweness_of_wavelet_transformed_image_continuos: float = Field(alias="skeweness")
    curtosis_of_wavelet_transformed_image_continuos: float = Field(alias="curtosis")
    entropy_of_image_continuos: float = Field(alias="entropy")
    true_class: int = Field(alias="class")

banknotes: list[BankNote]  = []


In [25]:
banknotes_df = pd.read_csv(
    DATASETT_PATH,
    names=["variance", "skeweness", "curtosis", "entropy", "class"]
)
banknotes = [BankNote(**row) for row in banknotes_df.to_dict(orient="records")]
banknotes_df

,variance,skeweness,curtosis,entropy,class
0,3.62160,8.66610,-2.8073,-0.44699,0
1,4.54590,8.16740,-2.4586,-1.46210,0
2,3.86600,-2.63830,1.9242,0.10645,0
3,3.45660,9.52280,-4.0112,-3.59440,0
4,0.32924,-4.45520,4.5718,-0.98880,0
...,...,...,...,...,...
1367,0.40614,1.34920,-1.4501,-0.55949,1
1368,-1.38870,-4.87730,6.4774,0.34179,1
1369,-3.75030,-13.45860,17.5932,-2.77710,1
1370,-3.56370,-8.38270,12.3930,-1.28230,1


In [26]:
feat_y="skeweness"
feat_x="variance"
X = banknotes_df[feat_x].values
Y = banknotes_df[feat_y].values
labels = banknotes_df["class"].values
N_BINS: int = 30
print("X_shape: %s and head: %s" % (X.shape, X[:10]))
print("Y_shape: %s and head: %s" % (Y.shape, Y[:10]))

X_shape: (1372,) and head: [3.6216  4.5459  3.866   3.4566  0.32924 4.3684  3.5912  2.0922  3.2032
 1.5356 ]
Y_shape: (1372,) and head: [ 8.6661  8.1674 -2.6383  9.5228 -4.4552  9.6718  3.0129 -6.81    5.7588
  9.1772]


### Get the borders of the buckets
since its 3 buckets in a list (2D) 
```sh
edge[0]         edge[1]          edge[2]          edge[3]
|------bin_0------|------bin_1------|------bin_2------|
```

In [27]:
X = np.asarray(X) # for typing issues, turn it to normal array from np.aarray
x_max = cast(float, X.max()) # this is basicly a "Trust me bro, i know this is a float"
x_min = cast(float, X.min())

x_edges = np.linspace(x_min, x_max, int(N_BINS) + 1)
x_edges

array([-7.0421 , -6.57987, -6.11764, -5.65541, -5.19318, -4.73095,
       -4.26872, -3.80649, -3.34426, -2.88203, -2.4198 , -1.95757,
       -1.49534, -1.03311, -0.57088, -0.10865,  0.35358,  0.81581,
        1.27804,  1.74027,  2.2025 ,  2.66473,  3.12696,  3.58919,
        4.05142,  4.51365,  4.97588,  5.43811,  5.90034,  6.36257,
        6.8248 ])

In [28]:
Y = np.asarray(Y)
y_max = cast(float, Y.max())
y_min = cast(float, Y.min())

y_edges = np.linspace(y_min, y_max, int(N_BINS) + 1)
y_edges

array([-13.7731    , -12.88227667, -11.99145333, -11.10063   ,
       -10.20980667,  -9.31898333,  -8.42816   ,  -7.53733667,
        -6.64651333,  -5.75569   ,  -4.86486667,  -3.97404333,
        -3.08322   ,  -2.19239667,  -1.30157333,  -0.41075   ,
         0.48007333,   1.37089667,   2.26172   ,   3.15254333,
         4.04336667,   4.93419   ,   5.82501333,   6.71583667,
         7.60666   ,   8.49748333,   9.38830667,  10.27913   ,
        11.16995333,  12.06077667,  12.9516    ])

### actually make the bins

In [29]:
x_bin = np.clip(np.digitize(X, x_edges) - 1, 0, 2) # force the bins to the clossest bin if it was in the boundary(something that np.digitilize does sometimes putting stuff in the phantom bin)
x_bin

array([2, 2, 2, ..., 2, 2, 2], shape=(1372,))

In [30]:
y_bin = np.clip(np.digitize(Y, y_edges) - 1, 0, 2)
y_bin

array([2, 2, 2, ..., 0, 2, 2], shape=(1372,))

In [31]:
cell_classes = {}
cell_counts_in_the_grid = {}

for i in range(N_BINS):
    for j in range(N_BINS):
        mask = (x_bin == i) & (y_bin == j)
        points_in_cell = labels[mask]
        count = Counter(points_in_cell)
        if len(count) > 0:
            majority = count.most_common(1)[0][0]
            cell_classes[(i,j)] = majority
            cell_counts_in_the_grid[(i, j)] = Counter(points_in_cell)
        else:
            cell_classes[(i, j)] = None
            cell_counts_in_the_grid[(i, j)] = None
            
print(cell_classes)
print(cell_counts_in_the_grid)


{(0, 0): None, (0, 1): None, (0, 2): np.int64(1), (0, 3): None, (0, 4): None, (0, 5): None, (0, 6): None, (0, 7): None, (0, 8): None, (0, 9): None, (0, 10): None, (0, 11): None, (0, 12): None, (0, 13): None, (0, 14): None, (0, 15): None, (0, 16): None, (0, 17): None, (0, 18): None, (0, 19): None, (0, 20): None, (0, 21): None, (0, 22): None, (0, 23): None, (0, 24): None, (0, 25): None, (0, 26): None, (0, 27): None, (0, 28): None, (0, 29): None, (1, 0): None, (1, 1): None, (1, 2): np.int64(1), (1, 3): None, (1, 4): None, (1, 5): None, (1, 6): None, (1, 7): None, (1, 8): None, (1, 9): None, (1, 10): None, (1, 11): None, (1, 12): None, (1, 13): None, (1, 14): None, (1, 15): None, (1, 16): None, (1, 17): None, (1, 18): None, (1, 19): None, (1, 20): None, (1, 21): None, (1, 22): None, (1, 23): None, (1, 24): None, (1, 25): None, (1, 26): None, (1, 27): None, (1, 28): None, (1, 29): None, (2, 0): np.int64(1), (2, 1): np.int64(1), (2, 2): np.int64(0), (2, 3): None, (2, 4): None, (2, 5): None, 

### Draw the cells

In [32]:
fig = go.Figure()
#cell_colors = {0: color_palette[0], 1: color_palette[4], None: color_palette[8]}
cell_colors = {0: "rgba(174, 214, 241, 0.5)", 1: "rgba(169, 223, 171, 0.5)", None: "rgba(240,240,240,0.3)"}
cell_colors

{0: 'rgba(174, 214, 241, 0.5)',
 1: 'rgba(169, 223, 171, 0.5)',
 None: 'rgba(240,240,240,0.3)'}

In [33]:
for i in tqdm(range(N_BINS), desc="Going though rows from top"):
    for j in tqdm(range(N_BINS), desc="Going through collumns from left"):
        x0, x1 = x_edges[i], x_edges[i + 1]
        y0, y1 = y_edges[j], y_edges[j + 1]
        
        cell_class = cell_classes[(i,j)]
        counts = cell_counts_in_the_grid.get((i,j), {})
        
        if counts is not None:
            count_str = "<br>".join([f"Class {k}: {v}" for k, v in sorted(counts.items())])
            label = f"-> Class {cell_class}<br>{count_str}"
        else:
            count_str = ""
            label = "Empty"
                
        hover_txt = f"Cell ({i},{j})<br>Assigned: Class {cell_class}<br>{count_str}"
        
        fig.add_shape(
            type="rect",
            x0=x0, x1=x1, y0=y0, y1=y1,
            fillcolor=cell_colors[cell_class],
            line=dict(color="black", width=1.5)
        )
        label = f"-> Class {cell_class}<br>{count_str}" if cell_class is not None else "Empty"
        fig.add_annotation(
            x=(x0 + x1) / 2,
            y=(y0 + y1) / 2,
            text=label,
            showarrow=False,
            font=dict(size=15),
            align="center"
        )
        fig.add_trace(go.Scatter(
            x=[(x0 + x1) / 2],
            y=[(y0 + y1) / 2],
            mode="markers",
            marker=dict(size=0.1, opacity=0),
            hovertemplate=hover_txt + "<extra><extra>",
            showlegend=False
        ))
        
 
class_styles = {
    0: dict(color=color_palette[3], name="Class 0 (fake)"),
    1: dict(color=color_palette[4], name="Class 1 (real)")
}

for cell_class, class_style in class_styles.items():
    mask = labels == cell_class
    fig.add_trace(go.Scatter(
        x=X[mask],
        y=Y[mask],
        mode="markers",
        marker=dict(color=class_style["color"], opacity=0.5, size=5),
        name=class_style["name"],
    ))    
    
fig.update_layout(
    title="Cubic Histogram Classifier (3x3 grid)",
    xaxis_title=feat_x,
    yaxis_title=feat_y,
    legend=dict(x=1.02, y=1),
    width=8000,
    height=7000,
)
fig.show()

        

Going though rows from top: 100%|██████████| 30/30 [01:23<00:00,  2.77s/it]
